# lj mass difference on siganl and background

In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
from helpers import *

In [2]:
# Channels
channels = [
    "bkg_2mulj_iso_disp_dphi_ljmdiff1"
    # "bkg_2mulj",
    # "bkg_2mulj_iso",
    # "bkg_2mulj_iso_disp",
    # "bkg_2mulj_iso_disp_dphi",
    # "bkg_2mulj_iso_disp_dphi_ljmp10",
    # "bkg_2mulj_iso_disp_dphi_ljmp15",
    # "bkg_2mulj_iso_disp_dphi_ljmp20",
    # "bkg_2mulj_iso_disp_dphi_ljmp30",
    # "bkg_2mulj_iso_disp_dphi_ljmp10_mass150",
]

In [3]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

<Client: 'tls://192.168.197.226:8786' processes=1 threads=1, memory=3.82 GiB>

In [4]:
runner = processor.Runner(
    executor=processor.DaskExecutor(client=client),      # for dask
    # executor=processor.FuturesExecutor(),              # for testing locally
    # executor=processor.IterativeExecutor(),              # for testing locally
    schema = llpnanoaodschema.LLPNanoAODSchema,
    chunksize=1000,
    # maxchunks=1,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(    channels,    ["BKG_study"],    unweighted_hist=True, verbose=False)

In [5]:
nfiles = -1
vr = "39"

In [8]:
# process DYJ
dyj_samples = {
    "dyj1": bkgdyj1,
    "dyj2": bkgdyj2,
}

for d, dy in dyj_samples.items():
    filesetd  = utilities.make_fileset(dy, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
    outd      = runner.run(filesetd, treename="Events", processor_instance=p)
    outd      = outd["out"]
    # print(f"Saving: bkg_{vr}_{d}.coffea")
    coffea.util.save(outd, f"outputs/bkg_{vr}_{d}.coffea")
    print(f"Finished: {d}: {dy}")

KeyboardInterrupt: 

In [ ]:
# process 4mu: 4-9
signal4mu = {
    # "fmu4": fmulxy4,
    # "fmu5": fmulxy5,
    # "fmu6": fmulxy6,
    # "fmu7": fmulxy7,
    "fmu8": fmulxy8,
    "fmu9": fmulxy9,
}
for name, sample in signal4mu.items():
    filesets = utilities.make_fileset(sample, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_4mu_v10.yaml")
    outs    = runner.run(filesets, treename="Events", processor_instance=p)
    outs    = outs["out"]
    coffea.util.save(outs, f"outputs/bkg_{vr}_{name}.coffea")
    print(f"Finished: {name}: {sample}")

In [ ]:
print("done with you")